In [3]:
import optimize
import numpy as np
import pandas as pd
import importlib

conversion_to_annual = {
    "1d": 252,
    "1wk": 52,
    "1mo": 12,
}
annual_risk_free = 0.02

In [44]:
importlib.reload(optimize)

<module 'optimize' from '/Users/linanpluimgmail.com/repos/portfolio_optimizing/optimize.py'>

Current portfolio

In [46]:
# My current portfolio
tickers = [
            "VWCE.DE", #Vanguard FTSE All-World UCITS ETF (USD) Accumulating
            "IUSN.DE" #iShares MSCI World Small Cap UCITS ETF
           ]
weights = np.array([0.074, 0.926])  
interval = "1mo"  # "1d", "1wk", or "1mo"
anualization_factor = conversion_to_annual[interval]

risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

current_gm = optimize.sample_geometric_mean(weights, returns)
current_sharpe = optimize.sharpe_ratio(weights, mu, cov, risk_free)

print(f'\nCurrent {interval} GM:', round(current_gm, 6))
print("Annualized GM:", round((1 + current_gm) ** anualization_factor - 1, 6))
print(f'Current {interval} Sharpe:', round(current_sharpe, 6))
print("Annualized Sharpe:", round(current_sharpe * np.sqrt(anualization_factor), 6))

[*********************100%***********************]  2 of 2 completed

min date: 2019-08-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
geometric_return    0.1013   0.1269
arithmetic_return   0.1121   0.1291
volatility          0.1742   0.1348

Current 1mo GM: 0.009879
Annualized GM: 0.125213
Current 1mo Sharpe: 0.228078
Annualized Sharpe: 0.790084


Optimization

In [37]:
# Settings for optimization
tickers = [
            "VWCE.DE", #Vanguard FTSE All-World UCITS ETF (USD) Accumulating
            "IUSN.DE", #iShares MSCI World Small Cap UCITS ETF
            "PPFB.DE", #iShares Physical Gold ETC
            "SXRS.DE", #iShares Diversified Commodity Swap UCITS ETF
            "SEC0.DE" #iShares MSCI Global Semiconductors UCITS ETF USD (Acc)
           ] 
interval = "1mo"  # "1d", "1wk", or "1mo"
start = "2021-07-01" # or none if you want the whole history
end = None # or none if you want the whole history

In [45]:
# Optimize
returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
    start = start,
    end = end
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

anualization_factor = conversion_to_annual[interval]
risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1
sharpe = optimize.maximize_sharpe_ratio(
    mu, cov, risk_free, anualization_factor
)
gm = optimize.maximize_sample_geometric_mean(
    returns, anualization_factor
)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")

print("\nPortfolio weights:")
print(weights.round(4))

print(f"\nAnnualized Optimal Sharpe:", round(sharpe["annual_sharpe"], 6))
print(f"Annualized Optimal GM:", round(gm["annual_gm"], 6))

[*********************100%***********************]  5 of 5 completed


min date: 2021-09-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  PPFB.DE  SEC0.DE  SXRS.DE  VWCE.DE
geometric_return    0.0778   0.2058   0.3097   0.1171   0.1119
arithmetic_return   0.0872   0.1992   0.3372   0.1238   0.1144
volatility          0.1567   0.1480   0.3707   0.1608   0.1269

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
IUSN.DE         0.0000        0.0
PPFB.DE         0.5564        0.0
SEC0.DE         0.1777        1.0
SXRS.DE         0.2658        0.0
VWCE.DE         0.0000        0.0

Annualized Optimal Sharpe: 1.556874
Annualized Optimal GM: 0.309704
